# Private Market Data Interview Practice - Answers

This notebook contains reference implementations and quick checks.
Try questions first before looking at these.

In [8]:
from collections import defaultdict, Counter
from datetime import datetime
from statistics import pstdev

## Shared Sample Data

In [1]:
cashflows = [
    {"fund": "F1", "date": "2026-01-15", "type": "call", "amount": 100.0},
    {"fund": "F1", "date": "2026-03-20", "type": "distribution", "amount": 35.0},
    {"fund": "F2", "date": "2026-02-10", "type": "call", "amount": 200.0},
    {"fund": "F2", "date": "2026-05-01", "type": "distribution", "amount": 40.0},
]

valuations = [
    {"fund": "F1", "asset": "A", "as_of": "2026-06-30", "value": 120.0, "ingested_at": "2026-07-01T08:00:00"},
    {"fund": "F1", "asset": "A", "as_of": "2026-06-30", "value": 122.0, "ingested_at": "2026-07-01T10:00:00"},
    {"fund": "F2", "asset": "B", "as_of": "2026-06-30", "value": 250.0, "ingested_at": "2026-07-01T09:30:00"},
]

positions = [
    {"fund": "F1", "asset": "A", "exposure": 120.0},
    {"fund": "F1", "asset": "C", "exposure": 80.0},
    {"fund": "F2", "asset": "B", "exposure": 250.0},
    {"fund": "F2", "asset": "D", "exposure": 50.0},
]

## Answer 1 - Net Cashflow by Fund

In [ ]:
def net_cashflow_by_fund(records):
    totals = defaultdict(lambda: {"call": 0.0, "distribution": 0.0, "net": 0.0})
    for r in records:
        fund = r["fund"]
        t = r["type"]
        amt = float(r["amount"])
        totals[fund][t] += amt
    for fund, bucket in totals.items():
        bucket["net"] = bucket["distribution"] - bucket["call"]
    return dict(totals)

r1 = net_cashflow_by_fund(cashflows)
assert r1["F1"]["net"] == -65.0
assert r1["F2"]["call"] == 200.0
r1

## Answer 2 - Keep Latest Valuation per Business Key

In [ ]:
def dedupe_latest_valuation(records):
    latest = {}
    for r in records:
        key = (r["fund"], r["asset"], r["as_of"])
        ts = datetime.fromisoformat(r["ingested_at"])
        if key not in latest or ts > latest[key][0]:
            latest[key] = (ts, r)
    return [v[1] for v in latest.values()]

r2 = dedupe_latest_valuation(valuations)
assert len(r2) == 2
assert any(x["value"] == 122.0 for x in r2)
r2

## Answer 3 - Compute TVPI and DPI

In [ ]:
def compute_private_equity_multiples(cashflow_records, latest_valuation_records):
    cf = net_cashflow_by_fund(cashflow_records)
    nav = defaultdict(float)
    for r in latest_valuation_records:
        nav[r["fund"]] += float(r["value"])

    out = {}
    funds = set(cf) | set(nav)
    for f in funds:
        paid_in = cf.get(f, {}).get("call", 0.0)
        distributed = cf.get(f, {}).get("distribution", 0.0)
        f_nav = nav.get(f, 0.0)
        dpi = (distributed / paid_in) if paid_in else None
        tvpi = ((distributed + f_nav) / paid_in) if paid_in else None
        out[f] = {
            "paid_in": paid_in,
            "distributed": distributed,
            "nav": f_nav,
            "dpi": dpi,
            "tvpi": tvpi,
        }
    return out

latest_vals = dedupe_latest_valuation(valuations)
r3 = compute_private_equity_multiples(cashflows, latest_vals)
assert round(r3["F1"]["tvpi"], 4) == round((35 + 122) / 100, 4)
r3

## Answer 4 - Reconcile Two Data Sources

In [ ]:
def reconcile_values(source_a, source_b, tolerance=1.0):
    index_b = {
        (r["fund"], r["asset"], r["as_of"]): float(r["value"])
        for r in source_b
    }

    mismatches = []
    for a in source_a:
        key = (a["fund"], a["asset"], a["as_of"])
        a_val = float(a["value"])
        if key not in index_b:
            mismatches.append({"key": key, "reason": "missing_in_b"})
            continue
        b_val = index_b[key]
        diff = abs(a_val - b_val)
        if diff > tolerance:
            mismatches.append({"key": key, "a": a_val, "b": b_val, "abs_diff": diff})
    return mismatches

src_a = [{"fund": "F1", "asset": "A", "as_of": "2026-06-30", "value": 122.0}]
src_b = [{"fund": "F1", "asset": "A", "as_of": "2026-06-30", "value": 120.7}]
r4 = reconcile_values(src_a, src_b, tolerance=1.0)
assert len(r4) == 1
r4

## Answer 5 - Top-K Exposures per Fund

In [ ]:
def top_k_exposures(records, k=2):
    by_fund = defaultdict(list)
    for r in records:
        by_fund[r["fund"]].append((r["asset"], float(r["exposure"])))

    out = {}
    for fund, rows in by_fund.items():
        out[fund] = sorted(rows, key=lambda x: x[1], reverse=True)[:k]
    return out

r5 = top_k_exposures(positions, k=1)
assert r5["F2"][0][0] == "B"
r5

## Answer 6 - Staleness Flagging

In [ ]:
def find_stale_records(records, snapshot_date, max_age_days):
    snapshot = datetime.fromisoformat(snapshot_date).date()
    stale = []
    for r in records:
        d = datetime.fromisoformat(r["as_of"]).date()
        age = (snapshot - d).days
        if age > max_age_days:
            stale.append({"record": r, "age_days": age})
    return stale

stale = find_stale_records(dedupe_latest_valuation(valuations), "2026-08-15", 30)
assert len(stale) == 2
stale

## Answer 7 - Rolling Volatility

In [ ]:
def rolling_volatility(prices, window=3):
    if len(prices) < window + 1:
        return []

    returns = []
    for i in range(1, len(prices)):
        prev_p = float(prices[i - 1])
        p = float(prices[i])
        returns.append((p / prev_p) - 1.0)

    vols = []
    for i in range(window - 1, len(returns)):
        w = returns[i - window + 1 : i + 1]
        vols.append(pstdev(w))
    return vols

r7 = rolling_volatility([100, 101, 99, 102, 103, 101, 100, 104], window=3)
assert len(r7) == 5
r7

## Answer 8 - Data Quality Summary

In [ ]:
def quality_report(records):
    required = ["fund", "asset", "as_of", "value"]
    null_critical = 0
    neg_not_allowed = 0
    keys = []

    for r in records:
        if any(r.get(k) is None for k in required):
            null_critical += 1
        if r.get("value") is not None and float(r["value"]) < 0:
            neg_not_allowed += 1
        keys.append((r.get("fund"), r.get("asset"), r.get("as_of")))

    counts = Counter(keys)
    duplicate_keys = sum(1 for _, c in counts.items() if c > 1)

    return {
        "row_count": len(records),
        "null_critical": null_critical,
        "duplicate_business_keys": duplicate_keys,
        "negative_values": neg_not_allowed,
    }

bad = valuations + [{"fund": None, "asset": "X", "as_of": "2026-06-30", "value": -1, "ingested_at": "2026-07-01T11:00:00"}]
r8 = quality_report(bad)
assert r8["null_critical"] == 1
assert r8["negative_values"] == 1
r8

## Interview Reflection Checklist

After solving, practice verbal answers for:
- complexity and scaling strategy
- data-quality controls
- batch vs streaming choices
- test strategy and observability